In [27]:
!mkdir -p ~/.kaggle && echo KGAT_47800c4cc1307c47f6e62757a0afe20f > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [28]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [29]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip",'r')
zip.extractall("/content")
zip.close()

In [30]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [31]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [32]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [33]:
len(age)

23708

In [34]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [35]:
df.shape

(23708, 3)

In [36]:
df.head()

,age,gender,img
0,58,0,58_0_3_20170119192023433.jpg.chip.jpg
1,23,1,23_1_3_20170117144834977.jpg.chip.jpg
2,27,0,27_0_0_20170104201504434.jpg.chip.jpg
3,32,0,32_0_1_20170117140225687.jpg.chip.jpg
4,16,0,16_0_0_20170110231230149.jpg.chip.jpg


In [37]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [38]:
train_df.shape

(20000, 3)

In [39]:
test_df.shape

(3708, 3)

In [40]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [41]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [42]:
import tensorflow as tf

def generator_wrapper(generator):
    while True:
        x, y = next(generator)
        yield x, tuple(y)

In [43]:
train_gen = generator_wrapper(train_generator)
test_gen = generator_wrapper(test_generator)

In [44]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [45]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [46]:
resnet.trainable = False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [47]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [48]:
model.compile(optimizer='adam',loss={'age':'mae','gender':'binary_crossentropy'},metrics={'age':'mae','gender':'accuracy'},loss_weights={'age':1,'gender':99})

In [49]:
model.fit(
    train_gen,
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=test_gen,
    validation_steps=len(test_generator)
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 244s 374ms/step - age_loss: 15.5481 - age_mae: 15.5481 - gender_accuracy: 0.5094 - gender_loss: 0.8885 - loss: 103.5071 - val_age_loss: 14.9760 - val_age_mae: 14.9788 - val_gender_accuracy: 0.5189 - val_gender_loss: 0.6930 - val_loss: 83.5821
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 223s 357ms/step - age_loss: 15.0242 - age_mae: 15.0242 - gender_accuracy: 0.5224 - gender_loss: 0.6955 - loss: 83.8757 - val_age_loss: 14.6983 - val_age_mae: 14.6983 - val_gender_accuracy: 0.5178 - val_gender_loss: 0.6925 - val_loss: 83.2587
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 223s 357ms/step - age_loss: 14.9241 - age_mae: 14.9241 - gender_accuracy: 0.5238 - gender_loss: 0.6937 - loss: 83.6052 - val_age_loss: 14.9939 - val_age_mae: 14.9939 - val_gender_accuracy: 0.5175 - val_gender_loss: 0.6926 - val_loss: 83.5584
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 218s 349ms/step - age_loss: 14.8708 - age_mae: 14.8708 - gender_accuracy: 0.5234 - gender_loss: 0.6931 - loss: 83